# Fake-data schematic charts for the model structure diagram

Generates illustrative "fake data" thumbnails used by `img/build_svg.py` in the
model structure diagram ([model_structure_diagram.ipynb](model_structure_diagram.ipynb)).

These are **not** real model outputs (see `img/sources/README.md` for the earlier
real-screenshot versions, kept for reference but no longer used). Each chart here
uses synthetic data with no tick numbers - only large, legible axis labels - so it
stays readable once shrunk into a small panel, while still making clear what *type*
of data the panel represents (a noisy time series, a seasonal wave, a declining cost
curve, ...).

Run this notebook top to bottom to (re)write the PNGs into `img/sources/`, then
re-run `img/build_svg.py` to rebuild the diagram.


In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np

import src.matplotlib_style  # noqa: F401

rng = np.random.default_rng(7)

OUT_DIR = os.path.join("img", "sources")
os.makedirs(OUT_DIR, exist_ok=True)

INK = "#1F2A44"
MUTED = "#4A5568"


def style_axes(ax, xlabel, ylabel, label_size=13):
    # Rotated y-labels are bounded by figure HEIGHT (not width) - a long word on a
    # short/wide figure will run off the top of the canvas. Keep y-labels short
    # (<=8 chars at this size); x-labels have the full figure width to work with.
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(INK)
        ax.spines[spine].set_linewidth(1.4)
    ax.set_xlabel(xlabel, fontsize=label_size, color=INK, labelpad=6)
    ax.set_ylabel(ylabel, fontsize=label_size, color=INK, labelpad=6)
    ax.grid(False)


def new_figure(figsize, **subplots_kwargs):
    # src.matplotlib_style turns on constrained_layout globally, which silently
    # overrides subplots_adjust (and can crush rotated axis labels together on
    # multi-panel figures). Disable it here so our explicit margins hold.
    fig, axes = plt.subplots(figsize=figsize, **subplots_kwargs)
    fig.set_layout_engine(None)
    return fig, axes


def save(fig, name):
    # No bbox_inches="tight": on rotated y-labels it sometimes crops a hair too
    # close and clips the first/last glyph. Margins are set explicitly per-figure
    # via fig.subplots_adjust(...) instead, so a plain save keeps the full label.
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=200)
    plt.close(fig)
    print("wrote", path)


## Step 1 panels

In [2]:
# Weather reanalysis -> daily solar/wind capacity factor: noisy fluctuating signal
fig, ax = new_figure((3.3, 1.35))
fig.subplots_adjust(left=0.18, right=0.97, top=0.94, bottom=0.28)
x = np.linspace(0, 1, 400)
signal = 0.5 + 0.28 * np.sin(2 * np.pi * 3.1 * x) * np.exp(-((x - 0.5) ** 2) / 0.5)
noise = rng.normal(0, 0.14, size=x.size)
y = np.clip(signal + noise, 0.02, 0.98)
ax.plot(x, y, color="#3B6FB0", linewidth=1.1)
ax.fill_between(x, 0, y, color="#3B6FB0", alpha=0.15)
ax.set_ylim(-0.05, 1.05)
style_axes(ax, "Time", "Capacity")
save(fig, "fake_weather.png")


wrote img\sources\fake_weather.png


C:\Users\Yutong Ren\AppData\Local\Temp\ipykernel_13176\3955725041.py:3: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(left=0.18, right=0.97, top=0.94, bottom=0.28)


In [3]:
# Historical demand + CB7 projections: smooth seasonal wave
fig, ax = new_figure((3.3, 1.35))
fig.subplots_adjust(left=0.18, right=0.97, top=0.94, bottom=0.28)
x = np.linspace(0, 1, 400)
seasonal = 0.55 + 0.3 * np.sin(2 * np.pi * 2 * x - 0.6)
trend = -0.12 * x
wobble = rng.normal(0, 0.02, size=x.size)
y = seasonal + trend + wobble
ax.plot(x, y, color="#3B6FB0", linewidth=1.6)
ax.set_ylim(y.min() - 0.15, y.max() + 0.15)
style_axes(ax, "Year", "Demand")
save(fig, "fake_demand.png")


wrote img\sources\fake_demand.png


C:\Users\Yutong Ren\AppData\Local\Temp\ipykernel_13176\3542015161.py:3: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(left=0.18, right=0.97, top=0.94, bottom=0.28)


In [4]:
# DESNZ technology cost data -> discounted-cash-flow LCOE: declining cost curve
fig, ax = new_figure((3.3, 1.35))
fig.subplots_adjust(left=0.14, right=0.97, top=0.94, bottom=0.28)
x = np.linspace(0.05, 1, 300)
y = 0.12 + 0.85 / (x * 6) ** 0.9
y = np.clip(y, 0, 1.6)
ax.plot(x, y, color="#3B6FB0", linewidth=1.8)
ax.scatter([x[-1]], [y[-1]], color="#3B6FB0", s=18, zorder=3)
ax.set_ylim(-0.05, 1.6)
style_axes(ax, "Capacity factor", "Cost")
save(fig, "fake_cost.png")


wrote img\sources\fake_cost.png


C:\Users\Yutong Ren\AppData\Local\Temp\ipykernel_13176\3751663104.py:3: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(left=0.14, right=0.97, top=0.94, bottom=0.28)


## Step 2 panel

In [5]:
# Daily net supply - demand: noisy signal oscillating around a zero baseline
fig, ax = new_figure((3.3, 1.25))
fig.subplots_adjust(left=0.18, right=0.97, top=0.90, bottom=0.30)
x = np.linspace(0, 1, 500)
base = 0.15 * np.sin(2 * np.pi * 5 * x)
noise = rng.normal(0, 0.45, size=x.size)
y = base + noise
ax.axhline(0, color=MUTED, linewidth=1, linestyle=(0, (3, 2)))
ax.plot(x, y, color="#4E8B5C", linewidth=0.8)
ax.set_ylim(-1.3, 1.3)
style_axes(ax, "Simulated day", "Balance")
save(fig, "fake_netsupply.png")


wrote img\sources\fake_netsupply.png


C:\Users\Yutong Ren\AppData\Local\Temp\ipykernel_13176\1633120269.py:3: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(left=0.18, right=0.97, top=0.90, bottom=0.30)


## Step 4 panel (enlarged)

In [6]:
# Step 4 "Time series" panel (enlarged): storage level + multi-channel energy flow
fig, (ax1, ax2) = new_figure(
    (7.2, 3.2), nrows=2, ncols=1, sharex=True, gridspec_kw={"height_ratios": [1, 1.3]}
)
fig.subplots_adjust(left=0.10, right=0.98, top=0.97, bottom=0.15, hspace=0.55)

x = np.linspace(0, 1, 1200)

# Top: storage level (%) - charges, repeatedly hits the ceiling, occasional deep draws
storage = 0.75 + 0.4 * np.sin(2 * np.pi * 18 * x)
storage += rng.normal(0, 0.12, size=x.size)
storage = np.clip(storage, 0, 1)
ax1.fill_between(x, 0, storage, color="#D98A3D", alpha=0.55, linewidth=0)
ax1.plot(x, storage, color="#8A4A22", linewidth=0.5)
ax1.set_ylim(0, 1.05)
ax1.set_xticks([])
ax1.set_yticks([])
for spine in ("top", "right", "bottom"):
    ax1.spines[spine].set_visible(False)
ax1.spines["left"].set_color(INK)
ax1.spines["left"].set_linewidth(1.4)
ax1.set_ylabel("Storage", fontsize=14, color=INK, labelpad=6)

# Bottom: several overlaid noisy energy-flow channels (imports / gas / DAC / curtailed)
channels = [
    ("#7C5A9E", 0.9, 0.35),
    ("#4E8B5C", 0.6, 0.22),
    ("#1F2A44", 0.35, 0.12),
]
for color, scale, noise_sd in channels:
    y = scale * (0.5 + 0.5 * np.sin(2 * np.pi * rng.uniform(6, 14) * x + rng.uniform(0, 6)))
    y = np.clip(y + rng.normal(0, noise_sd, size=x.size), 0, None)
    ax2.plot(x, y, color=color, linewidth=0.7, alpha=0.85)
ax2.set_ylim(0, 1.5)
style_axes(ax2, "Simulated day (40 years)", "Energy flow", label_size=14)

save(fig, "fake_timeseries.png")


wrote img\sources\fake_timeseries.png


C:\Users\Yutong Ren\AppData\Local\Temp\ipykernel_13176\264388067.py:5: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(left=0.10, right=0.98, top=0.97, bottom=0.15, hspace=0.55)
